In [1]:
import os
import time
import re
import sqlite3
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Конфигурация
BASE_URL = "https://elib.uraic.ru"
START_URL = (
    "https://elib.uraic.ru/simple-search"
    "?location=%2F&query=&rpp=30&sort_by=score&order=desc"
    "&filter_field_1=dateIssued&filter_type_1=equals&filter_value_1=1990"
    "&filter_field_2=title&filter_type_2=contains&filter_value_2=%D0%9F%D1%80%D0%B0%D0%B2%D0%B4%D0%B0+%D0%BA%D0%BE%D0%BC%D0%BC%D1%83%D0%BD%D0%B8%D0%B7%D0%BC%D0%B0"
)
DOWNLOAD_FOLDER = "newspapers"
DB_PATH = "newspapers.db"
DELAY = 1

os.makedirs(DOWNLOAD_FOLDER, exist_ok=True)

# Словарь месяцев
MONTHS = {
    'янв': '01', 'фев': '02', 'мар': '03', 'апр': '04', 'май': '05', 'июн': '06',
    'июл': '07', 'авг': '08', 'сен': '09', 'окт': '10', 'ноя': '11', 'дек': '12'
}

def init_db():
    """Создаёт таблицу newspaper_mentions, если её нет."""
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute('''
        CREATE TABLE IF NOT EXISTS newspaper_mentions (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            newspaper_name TEXT NOT NULL,
            publication_date TEXT NOT NULL,
            person_name TEXT,
            context TEXT
        )
    ''')
    conn.commit()
    conn.close()

def save_mention_to_db(newspaper_name, publication_date, person_name=None, context=None):
    """Сохраняет запись в таблицу newspaper_mentions."""
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    try:
        cur.execute('''
            INSERT INTO newspaper_mentions 
            (newspaper_name, publication_date, person_name, context)
            VALUES (?, ?, ?, ?)
        ''', (newspaper_name, publication_date, person_name, context))
        conn.commit()
        print(f"  Запись добавлена в БД: {newspaper_name}, {publication_date}")
    except sqlite3.Error as e:
        print(f"  Ошибка БД: {e}")
    finally:
        conn.close()

def get_max_items():
    """Запрашивает у пользователя лимит."""
    while True:
        try:
            user_input = input("Сколько газет скачать? (0 или Enter - все): ").strip()
            if user_input == "" or user_input == "0":
                return None
            max_items = int(user_input)
            if max_items > 0:
                return max_items
            print("Введите положительное число или 0.")
        except ValueError:
            print("Введите целое число.")

def get_soup(url):
    """Загружает страницу и возвращает BeautifulSoup."""
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    try:
        resp = requests.get(url, headers=headers, verify=False, timeout=10)
        resp.raise_for_status()
        return BeautifulSoup(resp.text, 'html.parser')
    except Exception as e:
        print(f"Ошибка загрузки {url}: {e}")
        return None

def extract_item_links(soup):
    """Находит ссылки на карточки газет."""
    urls = []
    for a in soup.select('td a[href^="/handle/"]'):
        href = a.get('href')
        if href:
            full = urljoin(BASE_URL, href)
            if full not in urls:
                urls.append(full)
    return urls

def extract_pdf_link(soup, item_url):
    """Ищет прямую ссылку на PDF."""
    # Прямая ссылка
    pdf = soup.select_one('a[href$=".pdf"]')
    if pdf:
        return urljoin(BASE_URL, pdf['href'])
    # Через предпросмотр
    view = soup.find('a', string=re.compile(r'Просмотреть/Открыть'))
    if view and view.get('href'):
        view_url = urljoin(BASE_URL, view['href'])
        print(f"    Переход на предпросмотр: {view_url}")
        time.sleep(DELAY)
        view_soup = get_soup(view_url)
        if view_soup:
            pdf = view_soup.select_one('a[href$=".pdf"]')
            if pdf:
                return urljoin(BASE_URL, pdf['href'])
    return None

def extract_metadata(soup):
    """Извлекает название и дату из карточки."""
    title = None
    date_str = None
    table = soup.find('table', class_='table')
    if not table:
        return None, None
    for row in table.find_all('tr'):
        cells = row.find_all('td')
        if len(cells) >= 2:
            label = cells[0].get_text(strip=True)
            value = cells[1].get_text(strip=True)
            if 'Название:' in label:
                title = value
            elif 'Дата публикации:' in label:
                date_str = value
    return title, date_str

def clean_title(raw):
    """Очищает название от лишних пробелов и точки в конце."""
    if not raw:
        return raw
    cleaned = re.sub(r'\s+', ' ', raw)
    return cleaned.rstrip('.').strip()

def format_date(date_str):
    """Преобразует '24-июл-1980' → '1980-07-24'."""
    if not date_str:
        return None
    match = re.match(r'(\d{1,2})-([а-я]{3})-(\d{4})', date_str.lower())
    if match:
        day, mon_abbr, year = match.groups()
        mon_num = MONTHS.get(mon_abbr)
        if mon_num:
            return f"{year}-{mon_num}-{day.zfill(2)}"
    return None

def get_next_page(soup, current_url):
    """Находит ссылку на следующую страницу."""
    next_link = soup.find('a', class_='next-page-link') or soup.find('a', string='Вперед >')
    if next_link and next_link.get('href'):
        return urljoin(current_url, next_link['href'])
    return None

def download_pdf(pdf_url, folder, filename_base):
    """Скачивает PDF и возвращает имя файла."""
    if not pdf_url:
        return None
    safe_name = "".join(c for c in filename_base if c.isalnum() or c in (' ', '-', '_'))
    safe_name = re.sub(r'\s+', ' ', safe_name).strip()
    local_path = os.path.join(folder, f"{safe_name}.pdf")
    if os.path.exists(local_path):
        print(f"  Файл уже существует: {safe_name}.pdf")
        return safe_name
    print(f"  Скачиваю: {safe_name}.pdf")
    headers = {'User-Agent': 'Mozilla/5.0'}
    try:
        with requests.get(pdf_url, stream=True, headers=headers, verify=False, timeout=30) as r:
            r.raise_for_status()
            with open(local_path, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
        print(f"  Сохранено: {safe_name}.pdf")
        return safe_name
    except Exception as e:
        print(f"  Ошибка скачивания: {e}")
        return None

def main():
    if not os.path.exists(DB_PATH):
        print(f"Файл БД '{DB_PATH}' не найден. Создаю таблицу newspaper_mentions...")
        init_db()
    else:
        init_db()  # гарантия, что таблица существует

    max_items = get_max_items()
    print(f"\nЛимит: {max_items if max_items else 'безлимит'}")

    current_url = START_URL
    page_num = 1
    processed = 0
    downloaded = 0

    while current_url:
        print(f"\n--- Страница {page_num}: {current_url} ---")
        soup = get_soup(current_url)
        if not soup:
            break

        item_links = extract_item_links(soup)
        print(f"Найдено ссылок: {len(item_links)}")

        for url in item_links:
            if max_items and processed >= max_items:
                print(f"\nДостигнут лимит {max_items}. Останов.")
                break

            processed += 1
            print(f"\n[{processed}] Обработка: {url}")

            card_soup = get_soup(url)
            if not card_soup:
                continue

            raw_title, raw_date = extract_metadata(card_soup)
            # Название газеты: если не найдено — используем фильтр
            if raw_title:
                newspaper_name = clean_title(raw_title)
            else:
                newspaper_name = "Правда коммунизма"  # из фильтра
                print(f"  Название не определено, использую '{newspaper_name}'")

            publication_date = format_date(raw_date)
            if publication_date:
                print(f"  Дата: {raw_date} → {publication_date}")
                filename_base = f"{publication_date}_{newspaper_name}"
            else:
                print(f"  Дата не распознана: {raw_date}")
                publication_date = "1900-01-01"  # заглушка
                filename_base = newspaper_name

            pdf_url = extract_pdf_link(card_soup, url)
            if pdf_url:
                saved_name = download_pdf(pdf_url, DOWNLOAD_FOLDER, filename_base)
                if saved_name:
                    downloaded += 1
                    # Сохраняем в БД (person_name и context пока пустые)
                    save_mention_to_db(
                        newspaper_name=newspaper_name,
                        publication_date=publication_date,
                        person_name=None,
                        context=None
                    )
            else:
                print("  PDF не найден")

            time.sleep(DELAY)

        if max_items and processed >= max_items:
            break

        next_url = get_next_page(soup, current_url)
        if next_url and next_url != current_url:
            current_url = next_url
            page_num += 1
            time.sleep(DELAY * 2)
        else:
            print("\nПагинация завершена.")
            break

    print(f"\n✅ Готово! Обработано: {processed}, скачано PDF: {downloaded}")

if __name__ == "__main__":
    main()

c:\Users\nikol\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(



Лимит: 10

--- Страница 1: https://elib.uraic.ru/simple-search?location=%2F&query=&rpp=30&sort_by=score&order=desc&filter_field_1=dateIssued&filter_type_1=equals&filter_value_1=1990&filter_field_2=title&filter_type_2=contains&filter_value_2=%D0%9F%D1%80%D0%B0%D0%B2%D0%B4%D0%B0+%D0%BA%D0%BE%D0%BC%D0%BC%D1%83%D0%BD%D0%B8%D0%B7%D0%BC%D0%B0 ---
Найдено ссылок: 30

[1] Обработка: https://elib.uraic.ru/handle/123456789/11518
  Дата: 6-янв-1990 → 1990-01-06
  Скачиваю: 1990-01-06_Правда коммунизма 1990 003.pdf
  Сохранено: 1990-01-06_Правда коммунизма 1990 003.pdf
  Запись добавлена в БД: Правда коммунизма. 1990. № 003, 1990-01-06

[2] Обработка: https://elib.uraic.ru/handle/123456789/11548
  Дата: 13-мар-1990 → 1990-03-13
  Скачиваю: 1990-03-13_Правда коммунизма 1990 031.pdf
  Сохранено: 1990-03-13_Правда коммунизма 1990 031.pdf
  Запись добавлена в БД: Правда коммунизма. 1990. № 031, 1990-03-13

[3] Обработка: https://elib.uraic.ru/handle/123456789/11546
  Дата: 6-мар-1990 → 1990-03-06
  С

In [1]:
import sqlite3

DB_PATH = "newspapers.db"

def print_all_mentions():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    cursor.execute("""
        SELECT 
            id, 
            newspaper_name, 
            publication_date, 
            person_name, 
            context,
            pdf_filename
        FROM newspaper_mentions 
        ORDER BY publication_date, newspaper_name, id
    """)
    rows = cursor.fetchall()
    
    
    if not rows:
        print("Таблица newspaper_mentions пуста.")
        conn.close()
        return
    
    print(f"{'ID':<5} {'Газета':<35} {'Дата':<12} {'Имя человека':<28} {'Контекст'}")
    print("-" * 130)
    
    for row in rows:
        id_, newspaper, date, person, context, pdf = row
        
        newspaper = newspaper or "Не указано"
        person = person or "—"
        context_snippet = (context[:55] + "...") if context and len(context) > 55 else (context or "—")
        
        # Обрезаем слишком длинные названия
        if len(newspaper) > 33:
            newspaper = newspaper[:30] + "..."
        
        print(f"{id_:<5} {newspaper:<35} {date:<12} {person:<28} {context_snippet}")
    
    conn.close()
    print(f"\nВсего записей в базе: {len(rows)}")


def show_unique_newspapers():
    """Показывает, какие газеты сейчас есть в базе"""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("""
        SELECT newspaper_name, COUNT(*) as count 
        FROM newspaper_mentions 
        GROUP BY newspaper_name 
        ORDER BY count DESC
    """)
    print("\n📊 Уникальные названия газет в базе:")
    for name, count in cursor.fetchall():
        print(f"   • {name} — {count} записей")
    conn.close()


if __name__ == "__main__":
    show_unique_newspapers()
    print("\n" + "="*80)
    print_all_mentions()


📊 Уникальные названия газет в базе:

Таблица newspaper_mentions пуста.


In [2]:
import sqlite3
import pandas as pd

DB_PATH = "newspapers.db"
EXCEL_FILE = "mentions.xlsx"

def export_to_excel():
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query("SELECT * FROM newspaper_mentions", conn)
    conn.close()
    
    if df.empty:
        print("Нет данных для экспорта.")
        return
    
    df.to_excel(EXCEL_FILE, index=False, engine='openpyxl')
    print(f"Экспортировано {len(df)} записей в {EXCEL_FILE}")

if __name__ == "__main__":
    export_to_excel()

Экспортировано 1421 записей в mentions.xlsx
